# Video Generation Fundamentals

**Module:** 18 — Video Generation

Why video is harder than images — temporal consistency, product modes, compute budgets, and evaluation axes.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Define text/image-to-video generation and its constraints
- Contrast clip-length models vs long-form pipelines
- List evaluation criteria unique to video (motion, flicker, identity)
- Estimate frames, megapixel-frames, and rough cost proxies
- Choose product modes: txt2video, img2video, video2video, extend, stylize


## What Makes Video Harder Than Images?

### Definition
Video generation must produce a **sequence** of frames that are individually good and **jointly coherent** in identity, lighting, physics, and camera motion — under much higher compute.

### Why it matters
Image tricks fail silently as flicker, morphing faces, and broken causality. Product promises about 'long cinematic shots' collide with model context and $ budgets.

### How it works
Models generate short clips (or latents across time) conditioned on text/image/video; longer stories are stitched via extend, keyframes, editing NLEs, and sometimes separate audio models.

### Intuition
Animation: every frame is a drawing, but the eye also judges the in-betweens.

### Pitfalls
- Judging only thumbnail frames
- Ignoring temporal identity drift
- Promising minute-long one-shot gens without a pipeline

### When to use
Any feature that turns prompts or stills into moving media.


### Product modes

| Mode | Input | Output job |
|------|-------|------------|
| **txt2video** | Prompt | Short clip from scratch |
| **img2video** | Still + prompt | Animate a keyframe |
| **video2video** | Clip + prompt/style | Restyle / transform |
| **Extend / continue** | Tail frames + prompt | Longer timelines |
| **Stylize** | Clip + style ref | Look transfer over time |

```mermaid
flowchart LR
  COND[Text / Image / Video] --> GEN[Video model]
  GEN --> CLIP[Short clip]
  CLIP --> EXT[Extend / Edit / Mux audio]
  EXT --> OUT[Deliverable]
```

### Why time explodes cost

```
frames = seconds × fps
work ∝ frames × width × height × denoise_steps   (rough)
```


In [ ]:
# Demo 1: video job math
from dataclasses import dataclass

@dataclass
class VideoJob:
    prompt: str
    seconds: float
    fps: int
    width: int
    height: int
    seed: int = 0
    steps: int = 30

    @property
    def frames(self) -> int:
        return int(self.seconds * self.fps)

    @property
    def megapixel_frames(self) -> float:
        return self.width * self.height * self.frames / 1e6

    def cost_proxy(self, price_per_mp_frame: float = 0.002) -> float:
        return self.megapixel_frames * price_per_mp_frame * (self.steps / 30)

job = VideoJob("drone shot over misty pines", seconds=4, fps=24, width=1280, height=720)
print(job.frames, "frames", f"{job.megapixel_frames:.1f} MP-frames", f"${job.cost_proxy():.2f} proxy")


In [ ]:
# Demo 2: clip-length vs long-form pipeline planner
def plan_duration(total_seconds: float, max_clip: float = 4.0, overlap: float = 0.5) -> dict:
    if total_seconds <= max_clip:
        return {"strategy": "single_clip", "clips": 1, "gen_seconds": total_seconds}
    step = max_clip - overlap
    n = int((total_seconds - overlap) // step) + 1
    return {
        "strategy": "extend_chain",
        "clips": n,
        "max_clip": max_clip,
        "overlap": overlap,
        "note": "plus NLE polish / audio stage",
    }

print(plan_duration(3))
print(plan_duration(18))


## Quality Axes for Video

### Definition
Beyond image axes, score **motion coherence**, **flicker**, **identity stability**, **camera continuity**, **physics plausibility**, and **audio sync** (if present).

### Why it matters
A gorgeous frame with boiling textures is a failed video.

### How it works
Maintain a golden prompt suite; rate clips with human + automated proxies (optical-flow smoothness, face embedding drift, CLIP frame–prompt scores).

### Intuition
Film dailies — watch playback, not contact sheets alone.

### Pitfalls
- Optimizing only the first frame
- No seed/model logging across extends
- Silent audio drift in 'finished' exports

### When to use
QA for every model upgrade and provider bakeoff.


| Axis | Failure example |
|------|-----------------|
| Motion coherence | Limbs teleport; gait jitters |
| Flicker | Texture sparkle frame-to-frame |
| Identity | Face morphs mid-shot |
| Camera continuity | Random jump cuts inside one gen |
| Prompt adherence | Missing requested action |
| Audio alignment | Footsteps off beat (multimodal) |

```
ASCII eval loop:
  generate -> playback 1x/0.5x -> score sheet -> archive seed/params -> accept/reject
```


In [ ]:
# Demo 3: temporal identity drift proxy
import math

def drift(series):
    # mean cosine drop between consecutive embedding vectors
    def cos(a, b):
        dot = sum(x*y for x,y in zip(a,b))
        na = math.sqrt(sum(x*x for x in a)); nb = math.sqrt(sum(y*y for y in b))
        return dot / (na*nb + 1e-9)
    scores = [cos(series[i], series[i+1]) for i in range(len(series)-1)]
    return {"mean_consec": sum(scores)/len(scores), "min": min(scores)}

stable = [[0.2,0.8,0.1],[0.21,0.79,0.11],[0.22,0.78,0.10]]
unstable = [[0.2,0.8,0.1],[0.9,0.1,0.2],[0.1,0.2,0.9]]
print("stable", drift(stable))
print("unstable", drift(unstable))


In [ ]:
# Demo 4: mock txt2video API shapes
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"
request = {
    "model": "sora-like-placeholder",
    "prompt": "A paper boat floats down a sunlit stream, gentle camera pan",
    "seconds": 4,
    "size": "1280x720",
}
response = {
    "id": "video_123",
    "status": "completed",
    "assets": [{"url": "https://example.invalid/v.mp4", "format": "mp4"}],
}
print("Authorization: Bearer", OPENAI_API_KEY[:8] + "...")
print(request)
print(response["status"], response["assets"][0]["format"])


## Safety Surface Over Time

### Definition
Video multiplies misuse vectors: actions unfold, audio may instruct, deepfakes gain credibility through motion.

### Why it matters
Frame-level filters miss temporal narratives (violence escalation, stalking sequences).

### How it works
Combine prompt filters, sampled-frame classifiers, action heuristics, likeness policy, and human review for high-risk tiers.

### Intuition
Moderating a film, not a poster.

### Pitfalls
- Only scanning frame 0
- No policy for real-person video likeness

### When to use
Consumer apps, ads, political content, any public sharing surface.


### Try it yourself — Fundamentals

1. Extend `VideoJob` with audio_fps and estimate muxed packet counts.
2. Build a rubric with weights for motion, identity, adherence, flicker.
3. Storyboard an 20s ad as 5×4s clips with overlap notes.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `MP-frames` | Megapixels × frame count — rough work proxy |
| `extend` | Continue a clip from its temporal boundary |
| `flicker` | High-frequency appearance change across frames |
| `img2video` | Animate from a still image condition |


### Workshop — Parameter journal — Video Fundamentals

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Video Fundamentals
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Video Fundamentals

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Video Fundamentals
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Video Fundamentals

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Video Fundamentals
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Video Fundamentals

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Video Fundamentals
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Video Fundamentals

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Video Fundamentals
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Video Fundamentals

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Video Fundamentals
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Video Fundamentals

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Video Fundamentals
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


### Workshop — Self-check — Video Fundamentals

Run this checklist printer and fill it honestly before moving on.


In [ ]:
# Workshop 8 — Video Fundamentals
for i,x in enumerate(['restated objectives','ran demos','named pitfall','named metric'],1):
    print(f'{i}. [ ] {x}')


## Key Takeaways

- Video multiplies image problems by time
- Short clips + editing pipelines beat naive long one-shots today
- Budget compute before product promises
- Evaluate motion and identity, not only stills
